# 面试问题：Agent 怎样实现 Human-in-the-loop 审批、中断与安全恢复？

可以直接复述的回答是：第一，审批不是一个布尔值，而是绑定 run、动作摘要、金额、策略版本和状态版本的票据。第二，产生副作用前要持久化 pause 事件。第三，恢复时重新校验票据签发人、有效期和动作摘要。第四，状态在等待期间发生变化时，旧审批必须失效。第五，拒绝和超时都要成为可重放状态。第六，用错误自动批准率、人工处理量和恢复轨迹比较方案。下面以售后退款 Agent 为例。

## 真实案例：电商退款审批

五条脱敏退款申请包含金额、账户年龄、证据和风险信号。目标是决定自动退款、请求人工审批或拒绝。事件和票据字段与真实工作流相似，但金额、账号和审批人均为教学数据；本例不连接支付系统。

In [1]:
refunds = [  # 定义五条具有真实审批语义的退款请求
    {"run_id": "RF-401", "amount": 39, "account_days": 800, "evidence": "物流确认未发货", "risk": "low"},  # 小额老账户且证据充分
    {"run_id": "RF-402", "amount": 480, "account_days": 420, "evidence": "商品破损照片", "risk": "medium"},  # 中额退款需要人工复核证据
    {"run_id": "RF-403", "amount": 120, "account_days": 2, "evidence": "仅文字说明", "risk": "high"},  # 新账户高风险申请
    {"run_id": "RF-404", "amount": 75, "account_days": 650, "evidence": "重复扣款流水", "risk": "low"},  # 小额且支付证据充分
    {"run_id": "RF-405", "amount": 1500, "account_days": 1100, "evidence": "签收后争议", "risk": "medium"},  # 大额争议必须人工审批
]  # 结束五条脱敏退款输入
print("退款输入：run | amount | account_days | risk | evidence")  # 展示审批器实际读取的业务字段
for refund in refunds:  # 逐条输出五个退款请求
    print(f"{refund['run_id']} | {refund['amount']:4} | {refund['account_days']:4} | {refund['risk']:6} | {refund['evidence']}")  # 用可读表格呈现风险上下文


退款输入：run | amount | account_days | risk | evidence
RF-401 |   39 |  800 | low    | 物流确认未发货
RF-402 |  480 |  420 | medium | 商品破损照片
RF-403 |  120 |    2 | high   | 仅文字说明
RF-404 |   75 |  650 | low    | 重复扣款流水
RF-405 | 1500 | 1100 | medium | 签收后争议


## Baseline / 基线：只按金额自动退款

常见起点是金额低于 500 元就自动批准。它延迟低，却会忽略新账户、高风险信号和证据质量。

In [2]:
def amount_baseline(refund):  # 实现只看退款金额的简单基线
    if refund["amount"] < 500:  # 小于固定阈值时直接自动退款
        return "auto_refund"  # 返回无需人工介入的动作
    return "manual_approval"  # 大额请求统一送人工审批
baseline_decisions = {refund["run_id"]: amount_baseline(refund) for refund in refunds}  # 对五条请求运行金额基线
print("金额基线：run | decision")  # 输出最简单方案的逐样本结果
for refund in refunds:  # 按输入顺序展示基线动作
    print(f"{refund['run_id']} | {baseline_decisions[refund['run_id']]}")  # 让高风险小额误批可见


金额基线：run | decision
RF-401 | auto_refund
RF-402 | auto_refund
RF-403 | auto_refund
RF-404 | auto_refund
RF-405 | manual_approval


## 核心实现：策略判定、持久化 Pause 与审批票据

高风险、新账户或中大金额进入人工审批；只有低风险、老账户且证据充分的小额申请自动退款。审批票据绑定动作摘要和状态版本。

In [3]:
import hashlib  # 使用摘要绑定审批票据与具体退款动作
import json  # 使用规范 JSON 生成稳定动作摘要
def policy_route(refund):  # 实现风险、账户年龄和金额联合审批策略
    if refund["risk"] == "high" or refund["account_days"] < 30:  # 新账户或高风险信号必须人工复核
        return "manual_approval", "账户或风险门禁"  # 返回审批路线和可解释原因
    if refund["amount"] > 200:  # 中大额退款需要人工确认证据
        return "manual_approval", "金额超过自动退款上限"  # 返回金额门禁原因
    return "auto_refund", "低风险、老账户且金额受限"  # 仅允许满足全部条件的小额请求自动执行
def action_digest(refund, state_version):  # 计算审批票据绑定的动作摘要
    action = {"run_id": refund["run_id"], "amount": refund["amount"], "state_version": state_version}  # 只纳入决定副作用身份的稳定字段
    payload = json.dumps(action, ensure_ascii=False, sort_keys=True)  # 规范序列化避免字段顺序改变摘要
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()  # 返回不可歧义的动作摘要
states = {}  # 保存五个运行的持久化教学状态
for refund in refunds:  # 对每条退款请求应用受治理策略
    route, reason = policy_route(refund)  # 计算当前请求的路线和原因
    version = 1  # 初始化退款状态版本
    status = "paused_for_approval" if route == "manual_approval" else "ready_to_refund"  # 在副作用前持久化暂停或就绪状态
    states[refund["run_id"]] = {"status": status, "version": version, "digest": action_digest(refund, version), "reason": reason}  # 保存恢复所需的审批上下文
print("受治理状态：run | status | version | digest | reason")  # 输出每条请求的中断决策与证据
for run_id, state in states.items():  # 逐条展示持久化状态
    print(f"{run_id} | {state['status']} | {state['version']} | {state['digest'][:10]} | {state['reason']}")  # 显示审批绑定信息


受治理状态：run | status | version | digest | reason
RF-401 | ready_to_refund | 1 | f12b96cd11 | 低风险、老账户且金额受限
RF-402 | paused_for_approval | 1 | 94dc7d2aa0 | 金额超过自动退款上限
RF-403 | paused_for_approval | 1 | 24cdcd7dad | 账户或风险门禁
RF-404 | ready_to_refund | 1 | fac1e6d14c | 低风险、老账户且金额受限
RF-405 | paused_for_approval | 1 | 7d8c89182a | 金额超过自动退款上限


## 失败案例与修正：等待期间金额变化使旧审批失效

RF-402 暂停后，客服把退款金额从 480 元改为 680 元。只校验 run_id 的天真实现仍会接受旧票据；安全恢复必须重新计算动作摘要并检查状态版本。

In [4]:
original = next(refund for refund in refunds if refund["run_id"] == "RF-402")  # 取出中额破损退款作为状态变化反例
old_state = dict(states["RF-402"])  # 保存暂停时的版本和动作摘要
old_ticket = {"ticket_id": "AP-9001", "run_id": "RF-402", "approved": True, "approver": "finance-oncall", "state_version": 1, "action_digest": old_state["digest"]}  # 构造绑定旧金额的审批票据
changed = dict(original)  # 复制退款请求模拟客服修改金额
changed["amount"] = 680  # 在等待审批期间将退款金额提高到 680 元
new_version = 2  # 状态变化后递增持久化版本
new_digest = action_digest(changed, new_version)  # 为新金额和版本生成新的动作摘要
naive_accept = old_ticket["approved"] and old_ticket["run_id"] == changed["run_id"]  # 天真恢复只检查批准标志和运行编号
safe_accept = old_ticket["approved"] and old_ticket["state_version"] == new_version and old_ticket["action_digest"] == new_digest  # 安全恢复同时校验版本和动作摘要
resume_action = "execute_refund" if safe_accept else "pause_and_request_new_approval"  # 根据完整票据校验决定下一步动作
print(f"失败行为：旧票据批准 480 元，当前金额 {changed['amount']} 元，天真接受={naive_accept}")  # 展示旧审批误用于新动作的风险
print(f"修正行为：old_version={old_ticket['state_version']}，current_version={new_version}，安全接受={safe_accept}")  # 展示版本不一致如何阻断恢复
print("恢复动作：", resume_action)  # 明确输出安全系统不会执行退款


失败行为：旧票据批准 480 元，当前金额 680 元，天真接受=True
修正行为：old_version=1，current_version=2，安全接受=False
恢复动作： pause_and_request_new_approval


## 结果表：金额基线与受治理策略对照

In [5]:
expected = {"RF-401": "auto_refund", "RF-402": "manual_approval", "RF-403": "manual_approval", "RF-404": "auto_refund", "RF-405": "manual_approval"}  # 定义五种业务场景的人工期望路线
baseline_errors = 0  # 初始化金额基线错误数
governed_errors = 0  # 初始化受治理策略错误数
print("run | expected | amount_baseline | governed")  # 输出逐请求路线对照表
for refund in refunds:  # 在同一批退款请求上比较两种方案
    baseline = amount_baseline(refund)  # 获取金额基线动作
    governed = policy_route(refund)[0]  # 获取风险约束后的动作
    baseline_errors += int(baseline != expected[refund["run_id"]])  # 累加基线路由错误
    governed_errors += int(governed != expected[refund["run_id"]])  # 累加核心策略错误
    print(f"{refund['run_id']} | {expected[refund['run_id']]} | {baseline} | {governed}")  # 展示错误来自哪条请求
print(f"错误路由数：baseline={baseline_errors}，governed={governed_errors}")  # 输出同一人工期望下的汇总对照
print(f"等待人工审批：{sum(state['status'] == 'paused_for_approval' for state in states.values())}/{len(states)}")  # 展示安全策略带来的人工负担


run | expected | amount_baseline | governed
RF-401 | auto_refund | auto_refund | auto_refund
RF-402 | manual_approval | auto_refund | manual_approval
RF-403 | manual_approval | auto_refund | manual_approval
RF-404 | auto_refund | auto_refund | auto_refund
RF-405 | manual_approval | manual_approval | manual_approval
错误路由数：baseline=2，governed=0
等待人工审批：3/5


## 结果解读

金额基线错误地自动批准 RF-403，因为它看不到新账户和高风险信号。受治理策略把三条请求持久化为 paused_for_approval，并明确给出原因。旧审批对修改后的 RF-402 无效，说明恢复安全依赖“批准了哪个动作”，而不是“这个 run 曾经被批准”。

## 生产边界

真实审批系统还需要审批人 RBAC、多签、过期时间、不可抵赖签名、Webhook 去重、通知升级和支付系统权威回读。策略版本与票据 schema 必须可迁移，审批页面应展示动作 diff。本例没有执行真实退款，也未衡量审批队列 SLA 和欺诈模型误报。

## 最小回归测试

In [6]:
assert len(refunds) >= 5  # 保证审批案例至少包含五条真实语义请求
assert states["RF-403"]["status"] == "paused_for_approval"  # 保证新账户高风险请求不会自动退款
assert baseline_errors > governed_errors  # 保证风险策略在同一教学集上减少错误路由
assert naive_accept is True  # 保证反例真实暴露只校验 run_id 的缺陷
assert safe_accept is False  # 保证旧审批不能恢复已变更金额的动作
assert resume_action == "pause_and_request_new_approval"  # 保证状态变化后重新请求人工审批
